<a href="https://colab.research.google.com/github/LayaHilarius/revolut-po-case-study/blob/main/Revolut_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install google-play-scraper


In [ ]:
import pandas as pd
from google_play_scraper import Sort, reviews

frames = []
for c in ['in', 'gb']:
    result, _ = reviews('com.revolut.revolut', lang='en', country=c, sort=Sort.NEWEST, count=2500)
    df = pd.DataFrame(result)
    df['country'] = c
    frames.append(df)

data = pd.concat(frames)
data = data[['reviewId','content','score','thumbsUpCount','at','appVersion','country']]
data.to_csv('reviews.csv', index=False)
print(len(data), data['country'].value_counts(), data['at'].min(), data['at'].max())


5000 country
in    2500
gb    2500
Name: count, dtype: int64 2026-10-07 13:25:27 2026-11-23 08:48:31


In [ ]:
for t in data[data.score == 1]['content'].head(20):
    print('-', t)


- why my transaction is still pending?
- The fact that even a metal client with very high net worth cannot get ahold of a human representative and is threatened with account freeze, because bots want some additional information, is utterly outrageous.
- Revolut does not protect its customer's interests. Revolut only wants to take, while the people working at Revolut do not have the knowledge or experience of real bank employees. With Revolut, it is easy to become a victim of fraud. Fortunately, there are plenty of real banks that actually protect their customers, so there is no reason to take the risk with this one. If you want your money to be secure, go to a real bank. With Revolut, it's easy to lose your money.
- it's not supported on Guatemala
- Can't update the app because it says my device is rooted even though it's not. Lazy AI slop app development.
- Absolutely terrible customer support. I've contacted Revolut multiple times about a registration issue, provided detailed explana

In [ ]:
import re, duckdb
data = data.drop_duplicates('reviewId')
themes = {
 'frozen_account': r'block|frozen|freez|suspend|restrict|account.{0,20}clos|clos.{0,20}account|locked',
 'verification':   r'verif|kyc|identity|document|selfie|face|scan|proof|source of funds',
 'support':        r'support|agent|chat|\bbot|human|no response|reply|complaint|customer service|talk to|\bhelp',
 'fees':           r'\bfees?\b|charg|commission|hidden|exchange rate',
 'transfers':      r'transfer|pending|\bsend|receiv|delay|stuck|refund|deposit|limit',
 'cards':          r'\bcard|declin|atm|apple pay|google pay',
 'app_access':     r'crash|bug|error|root|update|login|log in|passcode|password|\bpin\b|access|won.?t open|not work|keeps closing|unstable|unreliable|graphene',
 'scam_fraud':     r'scam|fraud|stole|stolen|hack|unauthori|cheat',
 'breach_outage':  r'breach|dark web|leak|outage|power failure|service down|\bdown\b',
}
text = data['content'].fillna('').str.lower()
for name, pattern in themes.items():
    data[name] = text.str.contains(pattern, regex=True)
data['untagged'] = ~data[list(themes)].any(axis=1)

query = " UNION ALL ".join([f"""
SELECT '{t}' AS theme, COUNT(*) AS reviews, ROUND(AVG(score),2) AS avg_rating,
  ROUND(100.0*SUM(CASE WHEN score<=2 THEN 1 ELSE 0 END)/COUNT(*),1) AS pct_negative,
  ROUND(100.0*SUM(CASE WHEN score<=2 THEN 1 ELSE 0 END)/(SELECT COUNT(*) FROM data WHERE score<=2),1) AS share_of_all_negative
FROM data WHERE {t}""" for t in themes])
print('Total reviews:', len(data), '| Angry (1-2 stars):', (data.score<=2).sum())
print(duckdb.sql(query + " ORDER BY share_of_all_negative DESC").df().to_string())

Total reviews: 2500 | Angry (1-2 stars): 685
            theme  reviews  avg_rating  pct_negative  share_of_all_negative
0      app_access      300        1.65          83.3                   36.5
1         support      270        2.38          63.7                   25.1
2       transfers      252        2.87          50.0                   18.4
3  frozen_account      111        1.32          91.9                   14.9
4    verification      114        1.83          78.1                   13.0
5           cards      161        3.06          47.2                   11.1
6      scam_fraud       64        1.86          75.0                    7.0
7   breach_outage       62        1.87          72.6                    6.6
8            fees       76        2.62          56.6                    6.3


In [ ]:
!pip install duckdb
import duckdb
themes_list = ['frozen_account','verification','support','fees','transfers','cards','app_bugs','scam_fraud']

query = " UNION ALL ".join([f"""
SELECT '{t}' AS theme,
       COUNT(*) AS reviews,
       ROUND(AVG(score), 2) AS avg_rating,
       ROUND(100.0 * SUM(CASE WHEN score <= 2 THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_negative,
       ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM data WHERE score <= 2), 1) AS share_of_all_negative
FROM data WHERE {t}""" for t in themes_list])

result = duckdb.sql(query + " ORDER BY reviews DESC").df()
print(result)

            theme  reviews  avg_rating  pct_negative  share_of_all_negative
0         support      270        2.38          63.7                   39.4
1       transfers      252        2.87          50.0                   36.8
2        app_bugs      196        1.62          84.2                   28.6
3           cards      161        3.06          47.2                   23.5
4    verification      114        1.83          78.1                   16.6
5  frozen_account      111        1.32          91.9                   16.2
6            fees       76        2.62          56.6                   11.1
7      scam_fraud       64        1.86          75.0                    9.3


In [ ]:
print(data[data.untagged]['score'].value_counts().sort_index())


score
1     412
2      70
3      80
4     266
5    2758
Name: count, dtype: int64


In [ ]:
locked = data.app_access | data.frozen_account | data.verification
angry = data.score <= 2
print('Locked-out share of angry reviews:', round(100 * (locked & angry).sum() / angry.sum(), 1), '%')
print('Of locked-out angry reviews, % also mention support:', round(100 * (locked & angry & data.support).sum() / (locked & angry).sum(), 1), '%')

Locked-out share of angry reviews: 53.3 %
Of locked-out angry reviews, % also mention support: 27.4 %


In [11]:
data.to_csv('reviews_2026-11-23.csv', index=False)
from google.colab import files
files.download('reviews_2026-11-23.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>